In [2]:
print("Hello World")

Hello World


1.Collect & Parse Multi-Source Data 

2.making unified chunking of data

3.Making Embeddings for chunks

4.Store in Vector DB

5.Retrieve & Generate Response


# Loading the data

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader,TextLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

loader=DirectoryLoader("../Single-Source/papers",glob="**/*.pdf",show_progress=True,loader_cls=PyMuPDFLoader)

docs=loader.load()
print(f"Loaded {len(docs)} documents")


100%|██████████| 7/7 [00:02<00:00,  2.78it/s]

Loaded 135 documents


# Reading All the Documents

In [5]:
def read_all_documents(directory):
    """ Reading All the documents from the given directory""" 
    all_docs=[]
    pdf_dir=Path(directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf files")
    for pdf in pdf_files:
        print(f"Reading {pdf.name}")
        try:
            loader=PyMuPDFLoader(str(pdf))
            docs=loader.load()
            for doc in docs:
                doc.metadata["source"]=pdf.name
                doc.metadata["file_type"]="pdf"
                all_docs.append(doc)
        except Exception as e:
            print(f"Error while reading the pdf {pdf_files} : {e}")
    return all_docs


In [6]:
doc=read_all_documents("../Single-Source/papers")

Found 7 pdf files
Reading 2104.14294v2.pdf
Reading 2301.08243v3.pdf
Reading 2506.09985v1.pdf
Reading Agenda-Workshop.pdf
Reading Applications_Standing_Out_GDG_UniDeb_Workshop.pdf
Reading Asfand_Yar_Presentation.pdf
Reading End2End_learning.pdf


# Chunks of processed Data

In [7]:
def split_into_chunks(documents,chunk_size=2000,chunk_overlap=300):
    """Splitting the documents into chunks""" 
    text_splitter=RecursiveCharacterTextSplitter(
        separators=["\n\n","\n"," "," "],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
        )
    documents_split=text_splitter.split_documents(documents)
    if documents_split:
        print(f"Successfully splitted the documents in chunks")
    return documents_split

In [8]:
chunks=split_into_chunks(doc)
print(f"Number of chunks: {len(chunks)}")

Successfully splitted the documents in chunks
Number of chunks: 324


# Make The Embeddings

In [9]:
## Lets do The Embedding

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


class Embeddings:
    """Make Embeddings for the chunks"""
    def __init__(self,model_name :str ="all-MiniLM-L6-v2"):
        """ Makes the embeddings for chunks
        Args:
            model_name(str): model name to be used for embeddings
        """
        self.model_name=model_name
        self.model=None
        self._initialize_model()
    
    def _initialize_model(self):
        """ Loads the embedding model""" 
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model {self.model_name} : {e}")
            raise

    def get_embeddings(self,texts:List[str]):
        """ Generate Embeddings for the given text
        Args:
            text(List[str]): list of text to be embedded
        Returns:
            np.ndarray: array of embeddings
        """
        if not self.model:
            raise ValueError("Model not initialized")
        try:
            embeddins=self.model.encode(texts,show_progress_bar=True)
            return embeddins
        except Exception as e:
            print(f"Error while generating the embeddings")
            raise 
        


In [10]:
texts = [d.page_content for d in doc]
emb=Embeddings()
emb.get_embeddings(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 141.98it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully: 384


Batches: 100%|██████████| 5/5 [00:15<00:00,  3.15s/it]


array([[-0.04262781, -0.05772876,  0.02217718, ...,  0.07850416,
        -0.01222124, -0.04740933],
       [ 0.00675796, -0.00454421,  0.02969245, ...,  0.0102286 ,
        -0.01432511, -0.0335126 ],
       [-0.03932101, -0.02087261,  0.02317989, ...,  0.03900884,
         0.05929421, -0.04264716],
       ...,
       [ 0.00240089, -0.07889684, -0.0513011 , ..., -0.05532934,
        -0.08837082, -0.00707085],
       [-0.0337984 , -0.03873516,  0.00615951, ..., -0.03710882,
        -0.08974063, -0.03468866],
       [ 0.02985242, -0.02577508,  0.02625614, ..., -0.06118382,
        -0.12558176, -0.0390336 ]], shape=(135, 384), dtype=float32)

# Vector Store

In [11]:
import chromadb
import uuid
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class VectorStore:
    """Storing the embeddings of documents and retrieving relevant documents for a query""" 
    def __init__(
        self,
        collection_name="rag_collection",
        persistent_dir="./multi_vector_store"):
        self.collection_name=collection_name
        self.persistent_dir=persistent_dir
        self.client=None
        self.collection=None
        self._initialize_client()

    def _initialize_client(self):
        """Initialize the chromadb client and collection"""
        os.makedirs(self.persistent_dir,exist_ok=True)
        self.client=chromadb.PersistentClient(
            path=self.persistent_dir
        )
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF documents embeddings for RAG"}
        )
        print(f"Initialized VectorStore with collection: {self.collection_name}")

    def add_documents(self,documents:List[Dict[str,Any]],embeddings:np.ndarray):
        """Add documents and their embeddings to the vector store"""
        if len(documents) !=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        ids=[]
        documents_text=[]
        embeddings_list=[]
        metadatas=[]
        for i,doc in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata=Dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embeddings.tolist())
        try:
            self.collection.add(
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list,
                ids=ids
            )
            print(f"Added {len(documents)} along with their embeddings {len(embeddings_list)}")
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise

    def query(self,query_embedding:np.ndarray ,top_k:int=5):
        """Query the vector store for relevant documents"""
        try:
            results=self.collection.query(
                query_embeddings=query_embedding,
                n_results=top_k,
                include=["metadatas","documents","distances"]
            )
            return results
        except Exception as e:
            print(f"Error querying vector store: {e}")
            raise



In [12]:
vector_store=VectorStore()

Initialized VectorStore with collection: rag_collection


# Retriever Augmented Generation

In [ ]:
class RetrieverAugmentedGeneration:
    def __init__(self,vectorstore=VectorStore,embedddings=Embeddings):
        self.vectorstore=vectorstore
        self.embeddings=embedddings
    def retrieve(self,query:str,top_k:int=3,threshold:float=0.01)-> List[Dict[str, Any]]:
        query_embedding=self.embeddings.get_embeddings([query])
        results=self.vectorstore.collection.query(
            query_embeddings=query_embedding,
            n_results=top_k,
            include=["metadatas","distances","documents"]
        )
        retrieved_docs=[]
        for i,(doc_id,distance,doc,metadata) in enumerate(zip(results["ids"][0],results["distances"][0],results["documents"][0],results["metadatas"][0])):
            similarity = 1.0 - distance
            print(f"Rank {i+1}: distance={distance:.4f}, similarity={similarity:.4f}")
            if similarity >= threshold:
                retrieved_docs.append({
                    "id":doc_id,
                    "documents":doc,
                    "metadata":metadata,
                    "similarity_score":similarity,
                    "distance":distance,
                    "rank":i+1,
                })
        return retrieved_docs

In [14]:
vector_store = VectorStore(
    collection_name="pdf_documents",
    persistent_dir="../updatedgenerativeai/vector_store"
)

Initialized VectorStore with collection: pdf_documents


In [15]:
embeddings = Embeddings()
vectorstore=VectorStore()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 218.90it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully: 384
Initialized VectorStore with collection: rag_collection


In [16]:
rag_retriever =RetrieverAugmentedGeneration(
    vectorstore= vectorstore,
    embedddings=embeddings
)


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.21it/s]

[]


# integration of retriever and generation

In [18]:
### simple RAG system pipeline

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(model_name="llama-3.3-70b-versatile", groq_api_key=os.getenv("GROQ_API_KEY"))

retriever = RetrieverAugmentedGeneration(vectorstore, embeddings)

In [19]:
## simple RAG function :retireve context + generate response

def reg_simple(query,retriever,llm,top_k=3):
    # retriever the context 
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['document'] for doc in results]) if results else "No context found"
    # generate response
    prompt = f"""Answer the following question based on the context provided:
    Context: {context}
    Question: {query}
    Answer: """

    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [26]:
answer = reg_simple("What is the transformer", retriever, llm)

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.92it/s]


In [27]:
answer

'The transformer is a type of neural network architecture introduced in 2017 by Vaswani et al. in the paper "Attention Is All You Need." It revolutionized the field of natural language processing (NLP) and has since been widely adopted in many areas of artificial intelligence (AI).\n\nThe transformer model is primarily designed for sequence-to-sequence tasks, such as machine translation, text summarization, and text generation. It relies entirely on self-attention mechanisms to process input sequences, eliminating the need for recurrent neural networks (RNNs) and convolutional neural networks (CNNs).\n\nThe key components of the transformer architecture include:\n\n1. **Self-Attention Mechanism**: This allows the model to weigh the importance of different input elements relative to each other. It enables the model to capture long-range dependencies and contextual relationships in the input sequence.\n2. **Encoder-Decoder Structure**: The transformer consists of an encoder and a decoder